In [1]:
import os
from dotenv import load_dotenv
import glob
from pathlib import Path
import tiktoken
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
#from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go



/var/folders/1d/ftjdsbz94nn31hsgfs13pp600000gn/T/ipykernel_49314/2494004845.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [ ]:
# Set the model
# Load the API Key

MODEL = "gpt-4.1-nano"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins sk-proj-


In [3]:
# read in all .md
knowledge = {}

filenames = glob.glob("../outputs/clean/*.md")

for filename in filenames:
    name = Path(filename).stem
    with open(filename, "r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()

In [4]:
knowledge.keys()

dict_keys(['geschäftskunden_strom_grundversorgung', 'privatkunden_service_zählerstand_mitteilen', 'störung', 'wissensdatenbank_freibad_waiblingen', 'privatkunden_service_abschläge_berechnen_verstehen', 'geschäftskunden_service_abrechnung_zahlung', 'geschäftskunden_erdgas_grundversorgung', 'netze_gasnetz_netznutzungsentgelte', 'privatkunden_waerme_mobile_heizzentralen_mieten', 'netze_gasnetz_netzanschluss', 'privatkunden_strom_grundversorgung', 'geschäftskunden_erdgas', 'privatkunden_strom_preisinformation', 'privatkunden_e-mobilitaet', 'privatkunden_baeder', 'geschäftskunden_dienstleistungen', 'karriere', 'privatkunden_baeder_nutzungsbedingungen_gaeste-wlan_baeder', 'privatkunden_service_umzugsservice', 'geschäftskunden_strom_preisinformation', 'privatkunden_strom', 'netze_stromnetz_anmeldung_e-ladestation', 'privatkunden_strom_stromkennzeichnung', 'kontakt', 'wissensdatenbank_tarifwechsel', 'wissensdatenbank_elektromobilitaet', 'netze_messstellenbetrieb', 'geschäftskunden_dienstleistu

In [5]:
# Load in everything in the knowledgebase using LangChain's loaders (instead of the old method)

documents = []
loader = DirectoryLoader("../outputs/clean", glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
folder_docs = loader.load()
for doc in folder_docs:
    doc.metadata["doc_type"] = Path(doc.metadata['source']).stem.split('_')[0]
    print(Path(doc.metadata['source']).stem.split('_')[0])
    documents.append(doc)

Geschäftskunden
Privatkunden
Störung
Wissensdatenbank
Privatkunden
Geschäftskunden
Geschäftskunden
Netze
Privatkunden
Netze
Privatkunden
Geschäftskunden
Privatkunden
Privatkunden
Privatkunden
Geschäftskunden
Karriere
Privatkunden
Privatkunden
Geschäftskunden
Privatkunden
Netze
Privatkunden
Kontakt
Wissensdatenbank
Wissensdatenbank
Netze
Geschäftskunden
Netze
Geschäftskunden
Privatkunden
Privatkunden
Privatkunden
Privatkunden
Geschäftskunden
Privatkunden
Netze
Netze
Wissensdatenbank
Netze
Wissensdatenbank
Netze
Privatkunden
Privatkunden
Privatkunden
Netze
Netze
Kundenportal
Geschäftskunden
Netze
Privatkunden
Wissensdatenbank
Wissensdatenbank
Privatkunden
Netze
Privatkunden
Unternehmen
Netze
Geschäftskunden
Netze
Netze
Geschäftskunden
Geschäftskunden
Geschäftskunden
Privatkunden
Netze
Netze
Wissensdatenbank
Service
Geschäftskunden
Aktuelles
Privatkunden
Geschäftskunden
Privatkunden
Privatkunden
Privatkunden
Netze
Geschäftskunden


In [58]:
documents[0]

Document(metadata={'source': '../outputs/clean/Geschäftskunden_Strom_Grundversorgung.md', 'doc_type': 'Geschäftskunden'}, page_content='# Geschäftskunden - Strom - Grundversorgung\nDie Grundversorgung ist die gesetzlich gesicherte Stromlieferung für Haushaltskunden, wenn kein anderer Stromvertrag abgeschlossen wurde – zuverlässig, transparent und jederzeit verfügbar.\n##  Downloads Strom Grundversorgung\n  *  Abwendungsvereinbarung Muster 2024 (PDF | 189 KB)\n  *  Ergänzende Bedingungen StromGVV (PDF | 36 KB)\n  *  Hinweise zur Grund- und Ersatzversorgung Strom - Oktober 2022 (PDF | 82 KB)\n  *  Preisblatt Grundversorgung Strom ab 01.01.2023 (PDF | 75 KB)\n  *  Preisblatt Grundversorgung Strom ab 01.01.2024 (PDF | 92 KB)\n  *  Preisblatt Grundversorgung Strom ab 01.01.2025 (PDF | 89 KB)\n  *  Preisblatt Grundversorgung Strom ab 01.01.2026 (PDF | 89 KB)\n  *  Preise Aufgrund Zahlungsverzug und Unterbrechung der Stromversorgung (PDF | 100 KB)\n\n##  Ersatzversorgung\nDie Ersatzversorgung

In [6]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 468 chunks
First chunk:

page_content='# Geschäftskunden - Strom - Grundversorgung
Die Grundversorgung ist die gesetzlich gesicherte Stromlieferung für Haushaltskunden, wenn kein anderer Stromvertrag abgeschlossen wurde – zuverlässig, transparent und jederzeit verfügbar.
##  Downloads Strom Grundversorgung
  *  Abwendungsvereinbarung Muster 2024 (PDF | 189 KB)
  *  Ergänzende Bedingungen StromGVV (PDF | 36 KB)
  *  Hinweise zur Grund- und Ersatzversorgung Strom - Oktober 2022 (PDF | 82 KB)
  *  Preisblatt Grundversorgung Strom ab 01.01.2023 (PDF | 75 KB)
  *  Preisblatt Grundversorgung Strom ab 01.01.2024 (PDF | 92 KB)
  *  Preisblatt Grundversorgung Strom ab 01.01.2025 (PDF | 89 KB)
  *  Preisblatt Grundversorgung Strom ab 01.01.2026 (PDF | 89 KB)
  *  Preise Aufgrund Zahlungsverzug und Unterbrechung der Stromversorgung (PDF | 100 KB)' metadata={'source': '../outputs/clean/Geschäftskunden_Strom_Grundversorgung.md', 'doc_type': 'Geschäftskunden'}


In [6]:
os.getcwd()

'/Users/sunzeyuan/projects/agent_exercises/crawler copy/retrieval'

In [10]:
# Pick an embedding model

#embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
db_name = "vector_db"

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
#vectorstore = Chroma(persist_directory=db_name, embedding_function=embeddings)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 468 documents


In [11]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 468 vectors with 1,536 dimensions in the vector store


# Visualize

In [12]:
# Prework

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
#colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]
main = {"Privatkunden": "blue", "Netze": "green",
        "Geschäftskunden": "red", "Wissensdatenbank": "orange"}
colors = [main.get(t, "lightgrey") for t in doc_types]

In [13]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [33]:
import sys; print(sys.executable)

/Users/sunzeyuan/projects/agent_exercises/crawler copy/.venv/bin/python


In [ ]:
MODEL = "gpt-4.1-nano"

encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(knowledge['geschäftskunden_strom_grundversorgung'])
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")


Total tokens for gpt-4.1-nano: 485
